# A gate that can actually say no

A data quality gate has one job and usually fails at it: deciding whether a dataset may proceed. Most gates warn, nothing is ever blocked, and the gate becomes a logging statement with a budget.

The structural reason is that schema checking and distribution checking get written as one pass, so a schema failure and a drift failure arrive as the same undifferentiated 'problem'. They are independent readings of one profile, and they should meet at one explicit decision.

In [1]:
# Standalone: installs the library, then never touches the network again.
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import NodeCandidate
from dataclasses import replace

def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         facets=None, deterministic=True):
    """A node manifest in one line. A real pack writes these as JSON."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic}, facets=dict(facets or {}))

print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The shape of this problem is already known

A template is a typed skeleton — every port declared, every slot empty. Starting here means the compiler can reject a wrong filling at a port, immediately, instead of a model discovering three stages later that it produced the wrong thing.

In [2]:
template = T.get("data.quality")
print(template.task, "\n")
for slot in template.slots:
    ins = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    outs = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    print(f"  {slot.id:<13} {ins:>34}  ->  {outs}"
          + ("   (optional)" if slot.optional else ""))

print("\nlayers:", template.skeleton().layers())
print("is a chain:", template.skeleton().is_chain)

Decide whether a dataset may proceed, with the reason recorded. 

  profile                                        —  ->  out:Profile
  schema                                in:Profile  ->  out:Findings
  distribution                          in:Profile  ->  out:Findings
  adjudicate    schema:Findings, distribution:Findings  ->  out:Verdict

layers: [['profile'], ['schema', 'distribution'], ['adjudicate']]
is a chain: False


## The mistakes people make in this shape

Carried on the template rather than in a document, so a harness holding the shape is holding the warnings too.

In [3]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. A gate that only ever warns. If nothing is ever blocked, the gate is a logging statement with a budget.

2. Thresholds chosen to make today's data pass. Choose them against a reference period, and record which one.



## Fill the slots

A slot is a contract. A candidate is one way to satisfy it. Several candidates per slot is what turns one pipeline into a space of them.

In [4]:
nodes = [
    node("dq.profile.pandas",   "data.profile",      [], [("out", "Profile")]),
    node("dq.profile.polars",   "data.profile",      [], [("out", "Profile")]),

    node("dq.schema.strict",    "check.schema",      [("in", "Profile")], [("out", "Findings")]),
    node("dq.schema.evolving",  "check.schema",      [("in", "Profile")], [("out", "Findings")],
         facets={"purpose.not_for": ["contracts with downstream consumers"]}),

    node("dq.dist.psi",         "check.distribution", [("in", "Profile")], [("out", "Findings")]),
    node("dq.dist.ks",          "check.distribution", [("in", "Profile")], [("out", "Findings")]),

    node("dq.gate.any",         "gate.decide",
         [("schema", "Findings"), ("distribution", "Findings")], [("out", "Verdict")]),
    node("dq.gate.weighted",    "gate.decide",
         [("schema", "Findings"), ("distribution", "Findings")], [("out", "Verdict")]),
]

filling = {
    "profile": ["dq.profile.pandas", "dq.profile.polars"],
    "schema": ["dq.schema.strict", "dq.schema.evolving"],
    "distribution": ["dq.dist.psi", "dq.dist.ks"],
    "adjudicate": ["dq.gate.any", "dq.gate.weighted"],
}

bench = replace(template.instantiate(filling), nodes=tuple(nodes))
print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 16


## The shape, drawn

Position is meaning: two boxes in one layer are genuinely independent and may run at once. Arrows carry the port they land on.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg313003-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Profile</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 2 parallel</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check schema</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="386" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Check distribution</text><text x="395" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Adjudicate</text><text x="721" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><path d="M246,141.0 C316.0,141.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg313003-arrow)"/><path d="M246,141.0 C316.0,141.0 316.0,182.0 386,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg313003-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg313003-arrow)"/><text x="642.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">schema</text><path d="M572,182.0 C642.0,182.0 642.0,141.0 712,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg313003-arrow)"/><text x="642.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">distribution</text></svg>', title='Data quality gate — shape', note='3 layers, widest 2. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

## Compile a route

Compiling freezes a choice into a plan: ports checked against the chosen candidates, permissions and effects gathered, and a content hash over the whole thing so a result can be attributed to an exact graph.

In [6]:
route = {"profile": "dq.profile.pandas", "schema": "dq.schema.strict",
         "distribution": "dq.dist.psi", "adjudicate": "dq.gate.any"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers        :", plan.layers)
print("parallel width:", plan.parallel_width)
print("deterministic :", plan.deterministic)
print("permissions   :", plan.permissions or "none")
print("effects       :", plan.effects or "none — nothing here touches the world")

plan:eff4c00ed6fea7833a985f889c3881eb
layers        : (('profile',), ('schema', 'distribution'), ('adjudicate',))
parallel width: 2
deterministic : True
permissions   : none
effects       : none — nothing here touches the world


## Break it on purpose

The check that earns its keep. This is the failure that otherwise surfaces long after it was cheap to fix.

In [7]:
# A "gate" that reads only the schema findings: one input, so not a gate at all.
one_eyed = replace(bench, edges=tuple(
    e for e in bench.wiring() if not (e.target == "adjudicate"
                                      and e.to_port == "distribution")))
problems = [p for p in one_eyed.validate() if "adjudicate" in p]
print("validator:", problems[0] if problems else "(nothing — which would be the bug)")

# And a candidate that cannot satisfy the slot it was placed in.
wrong = node("dq.schema.chatty", "check.schema",
             [("in", "Profile")], [("out", "Prose")])      # Prose, not Findings
sab = replace(bench, nodes=bench.nodes + (wrong,),
              candidates=bench.candidates + (NodeCandidate(id="dq.schema.chatty",
                                                           node_id="dq.schema.chatty"),),
              stages=tuple(replace(s, candidates=s.candidates + ("dq.schema.chatty",))
                           if s.id == "schema" else s for s in bench.stages))
try:
    compile_route(sab, {**route, "schema": "dq.schema.chatty"})
except CompileError as exc:
    for problem in exc.problems:
        print("refused:", problem)

validator: sub-step 'adjudicate' needs input 'distribution' (Findings) and no edge supplies it
refused: schema: 'dq.schema.chatty' does not produce 'Findings' on port 'out' — it gives ['Prose']


## What was actually explored

The honest counter. Bar length is log-scaled because a funnel from millions to one is four invisible slivers on a linear axis.

In [8]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      max(1, bench.route_count() // 3)),
    ("policy-eligible", max(1, bench.route_count() // 12)),
    ("evaluated",       min(24, max(2, bench.route_count() // 40))),
    ("chosen",          1),
], title="what the search actually looked at")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">16</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="423.7" height="26" rx="4" fill="#2d6cb5" opacity="0.54" stroke="#2d6cb5" stroke-width="1"/><text x="623.7" y="129" font-size="11" fill="#22303f">5</text><text x="687.7" y="129" font-size="10" fill="#68737f">÷3.2</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="163.9" height="26" rx="4" fill="#2d6cb5" opacity="0.34" stroke="#2d6cb5" stroke-width="1"/><text x="363.9" y="175" font-size="11" fill="#22303f">1</text><text x="427.9" y="175" font-size="10" fill="#68737f">÷5</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="259.8" height="26" rx="4" fill="#2d6cb5" opacity="0.41" stroke="#2d6cb5" stroke-width="1"/><text x="459.8" y="221" font-size="11" fill="#22303f">2</text><text x="523.8" y="221" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="163.9" height="26" rx="4" fill="#1f8a4c" opacity="0.34" stroke="#1f8a4c" stroke-width="1"/><text x="363.9" y="267" font-size="11" fill="#22303f">1</text><text x="427.9" y="267" font-size="10" fill="#68737f">÷2</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what the search actually looked at', note='Every row is a real filter, in order.', width=1000, height=342)

## Where the evidence pointed

Per-step outcomes, in bits. A route that failed tells you one bit: something was wrong. Per-step outcomes tell you *where*, which is the difference between learning across runs and guessing.

In [9]:
viz.evidence({
    "profile":    0.7,
    "schema":     1.6,    # caught a renamed column
    "distribution": -2.2, # thresholds were tuned to make today's data pass
    "adjudicate": 0.3,
}, title="data quality gate — bits per step")

Figure(svg='<svg viewBox="0 0 940 218" width="940" height="218" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="188" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="204" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">profile</text><rect x="525.0" y="60" width="103.4" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="636.4" y="73" font-size="10" text-anchor="start" fill="#68737f">+0.70</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">schema</text><rect x="525.0" y="90" width="236.4" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="769.4" y="103" font-size="10" text-anchor="start" fill="#68737f">+1.60</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">distribution</text><rect x="200.0" y="120" width="325.0" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="192.0" y="133" font-size="10" text-anchor="end" fill="#68737f">-2.20</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">adjudicate</text><rect x="525.0" y="150" width="44.3" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="577.3" y="163" font-size="10" text-anchor="start" fill="#68737f">+0.30</text></svg>', title='data quality gate — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=218)

## What this bought

The two checks are **structurally** independent, so a schema failure and a drift failure cannot be confused. The decision is a real join with two inputs, and removing one is caught rather than silently producing a gate that reads half the evidence.

---

Source, and the other notebooks in this series: [https://github.com/aidonerightcorp/browsergraph](https://github.com/aidonerightcorp/browsergraph)